# Validação 15 — Resposta explicável ao usuário

## Goal

Executar o fluxo científico completo e transformar a síntese em uma resposta conservadora e auditável. A conclusão deve separar direção do sinal, força do conjunto, confiança metodológica, limitações e fontes.

## Setup

O exemplo usa uma alegação em português e uma revisão sistemática real. O relatório é montado por regras determinísticas: o modelo de linguagem natural classifica relações entre textos, mas não redige nem decide a conclusão final.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import os
import sys

from IPython.display import Markdown, display

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from fatofake import (
    DEFAULT_EMBEDDING_MODEL, DEFAULT_NLI_MODEL, AMSTAR2_SOURCE_URL,
    AmstarConfidence, AmstarJudgment, AmstarRating, Bm25Index,
    ChunkingConfig, ClassificationConfig, ClinicalTrialsClient,
    CrossrefClient, DataCiteClient, EvidenceDirection, EvidenceStrength,
    ExtractionConfig, HybridIndex, PmcClient, PubMedClient, QualityLevel,
    ReportConclusion, ReportSource, SemanticIndex, SentenceTransformerEncoder,
    TransformersNliClassifier, assess_amstar2, build_claim_evidence_pairs,
    chunk_article_content, classify_claim_evidence_pairs,
    extract_evidence_statements, generate_evidence_report,
    methodology_summary_from_amstar, prepare_search_plan,
    quality_profile_from_amstar, render_evidence_report_markdown,
    retrieve_article_content, search_pubmed, synthesize_evidence,
    validate_analysis_input, validate_article_quality,
)

print(f'Execução UTC: {datetime.now(timezone.utc).isoformat()}')

Execução UTC: 2026-09-25T12:45:34.272902+00:00


## Steps

### 1. Recuperar e classificar as evidências

Reutilizamos as etapas já validadas: PubMed/PMC, chunks, ranking híbrido, extração das afirmações e classificação NLI.

In [2]:
class PmidQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ['33431520[pmid]']

claim = 'Beber café pode alterar o risco de câncer de próstata.'
analysis_input = validate_analysis_input(claim)
search_plan = prepare_search_plan(analysis_input, PmidQueryPlanner())
publication = search_pubmed(
    search_plan,
    PubMedClient(email=os.getenv('NCBI_EMAIL'), api_key=os.getenv('NCBI_API_KEY')),
    max_results_per_query=1,
).publications[0]
content = retrieve_article_content(
    publication,
    PmcClient(email=os.getenv('NCBI_EMAIL'), api_key=os.getenv('NCBI_API_KEY')),
)
chunks = chunk_article_content(content, ChunkingConfig(max_words=120, overlap_words=20))
lexical_index = Bm25Index(chunks)
semantic_index = SemanticIndex(
    chunks,
    SentenceTransformerEncoder(os.getenv('EMBEDDING_MODEL', DEFAULT_EMBEDDING_MODEL)),
)
hybrid_results = HybridIndex(lexical_index, semantic_index).search(claim, top_k=8)
statements = extract_evidence_statements(
    claim, hybrid_results,
    ExtractionConfig(max_statements=6, max_per_chunk=2, min_words=8, max_words=80),
)
pairs = build_claim_evidence_pairs(claim, statements)
assessments = classify_claim_evidence_pairs(
    pairs,
    TransformersNliClassifier(os.getenv('NLI_MODEL', DEFAULT_NLI_MODEL)),
    ClassificationConfig(minimum_confidence=0.60, minimum_margin=0.10),
)
print(f'PMID {publication.pmid}: {len(assessments)} relações classificadas.')

/private/tmp/fatofake-notebook-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22883.88it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 20388.98it/s]

PMID 33431520: 6 relações classificadas.


### 2. Aplicar a validação externa e o AMSTAR 2

Os julgamentos da etapa 14 são reconstruídos com sua proveniência. A aplicação é preliminar e aproximada porque o artigo sintetiza estudos observacionais de exposição, enquanto o AMSTAR 2 foi concebido para revisões de intervenções.

In [3]:
quality_report = validate_article_quality(
    publication, content, CrossrefClient(email=os.getenv('CROSSREF_EMAIL')),
    DataCiteClient(), ClinicalTrialsClient(),
)
judgment_data = [
    (1, AmstarRating.YES, 'Critérios definem população, exposição, comparação e desfecho.', 'Methods'),
    (2, AmstarRating.NO, 'Não foi localizado protocolo prévio ou registro.', 'Methods'),
    (3, AmstarRating.YES, 'A escolha de coortes é justificada.', 'Methods/Discussion'),
    (4, AmstarRating.PARTIAL_YES, 'Três bases e referências foram consultadas, mas a cobertura foi restrita.', 'Methods/Discussion'),
    (5, AmstarRating.YES, 'Seleção realizada independentemente por dois pesquisadores.', 'Methods'),
    (6, AmstarRating.YES, 'Extração realizada independentemente por dois pesquisadores.', 'Methods'),
    (7, AmstarRating.NO, 'Não há lista individual dos estudos excluídos.', 'Results'),
    (8, AmstarRating.YES, 'Os estudos incluídos são descritos em detalhe.', 'Methods/Results'),
    (9, AmstarRating.PARTIAL_YES, 'Newcastle–Ottawa não cobre integralmente os domínios requeridos.', 'Methods'),
    (10, AmstarRating.NO, 'Não foi localizado financiamento de cada estudo primário.', 'Article and supplements'),
    (11, AmstarRating.YES, 'Métodos de síntese, heterogeneidade e sensibilidade são relatados.', 'Methods'),
    (12, AmstarRating.NO, 'Não foi localizada análise do impacto do risco de viés na síntese.', 'Methods/Results'),
    (13, AmstarRating.YES, 'As limitações são consideradas na interpretação.', 'Discussion'),
    (14, AmstarRating.YES, 'A heterogeneidade é quantificada, explorada e discutida.', 'Methods/Discussion'),
    (15, AmstarRating.YES, 'Viés de publicação é investigado e discutido.', 'Methods/Results'),
    (16, AmstarRating.YES, 'Financiamento e conflitos da revisão são declarados.', 'Back matter'),
]
judgments = [
    AmstarJudgment(item_id=i, rating=r, evidence=e, section=s, source_url=content.pmc_url)
    for i, r, e, s in judgment_data
]
applicability_note = (
    'O AMSTAR 2 foi aplicado de modo aproximado: a revisão trata uma exposição '    'observacional, e a classificação precisa ser confirmada com instrumento adequado '    'a revisões etiológicas.'
)
amstar = assess_amstar2(
    judgments, reviewer='Avaliação preliminar do pipeline Fato ou Fake',
    single_reviewer=True, applicability_note=applicability_note,
)
methodological_profile = quality_profile_from_amstar(publication, quality_report, amstar)
print(f'AMSTAR 2: {amstar.confidence.value}; perfil: {methodological_profile.level.value}.')

AMSTAR 2: CRITICALLY_LOW; perfil: LOW.


### 3. Sintetizar e gerar a resposta

A direção encontrada pelo classificador não é apresentada como probabilidade de a alegação estar correta. Como há somente um artigo independente, a força permanece insuficiente, mesmo se o sinal textual for de apoio.

In [4]:
synthesis = synthesize_evidence(assessments, [methodological_profile])
methodology = methodology_summary_from_amstar(
    publication.pmid, amstar, methodological_profile.level
)
sources = [
    ReportSource('Artigo no PubMed', publication.url, pmid=publication.pmid),
    ReportSource('Texto completo no PubMed Central', content.pmc_url),
    ReportSource('Registro DOI', f'https://doi.org/{publication.doi}'),
    ReportSource('Instrumento AMSTAR 2', AMSTAR2_SOURCE_URL),
]
report = generate_evidence_report(
    claim, synthesis, [methodology], sources,
    additional_limitations=[
        'Os estudos primários são observacionais: associação não estabelece causalidade.',
        'Não localizar retratação ou conjunto de dados nas bases consultadas não prova sua ausência.',
    ],
)
rendered = render_evidence_report_markdown(report)
display(Markdown(rendered))

## Conclusão: Evidência insuficiente

A análise encontrou um sinal de apoio à alegação, mas 1 artigo independente não fornece evidência suficiente para uma conclusão segura.

### O que as evidências sugerem

- Direção do sinal: **apoio à alegação**.
- Probabilidades médias do classificador textual — não representam a probabilidade de a alegação estar correta: apoio 61.6%, contradição 3.8% e neutro 34.6%.
- Artigos independentes analisados: **1**.

### Confiança da análise

- Força do conjunto: **insuficiente**.
- PMID 33431520: confiança metodológica **criticamente baixa** pelo AMSTAR 2; qualidade usada na síntese: **baixa**.

### Por que a confiança é limitada

- Foi analisado apenas 1 artigo independente.
- O artigo PMID 33431520 possui 4 domínios críticos incompletos no AMSTAR 2: (2, 4, 7, 9).
- A avaliação metodológica do PMID 33431520 foi preliminar e feita por um único revisor.
- O AMSTAR 2 foi aplicado de modo aproximado: a revisão trata uma exposição observacional, e a classificação precisa ser confirmada com instrumento adequado a revisões etiológicas.
- Os estudos primários são observacionais: associação não estabelece causalidade.
- Não localizar retratação ou conjunto de dados nas bases consultadas não prova sua ausência.

### Fontes

- [Artigo no PubMed](https://pubmed.ncbi.nlm.nih.gov/33431520/)
- [Texto completo no PubMed Central](https://pmc.ncbi.nlm.nih.gov/articles/PMC7805365/)
- [Registro DOI](https://doi.org/10.1136/bmjopen-2020-038902)
- [Instrumento AMSTAR 2](https://www.bmj.com/content/358/bmj.j4008)

## Checks

As verificações garantem que a regra conservadora prevalece sobre o sinal direcional, que a baixa confiança metodológica chega à resposta e que todas as fontes permanecem visíveis.

In [5]:
assert synthesis.direction is EvidenceDirection.SUPPORTS
assert synthesis.strength is EvidenceStrength.INSUFFICIENT
assert synthesis.article_count == 1
assert amstar.confidence is AmstarConfidence.CRITICALLY_LOW
assert methodological_profile.level is QualityLevel.LOW
assert report.conclusion is ReportConclusion.INSUFFICIENT_EVIDENCE
assert len(report.sources) == len({source.url for source in report.sources}) == 4
assert all(source.url in rendered for source in report.sources)
assert 'não representam a probabilidade' in rendered
assert 'associação não estabelece causalidade' in rendered.lower()
assert 'aplicado de modo aproximado' in rendered.lower()
assert 'verdadeiro' not in rendered.lower() and 'falso' not in rendered.lower()

print(
    'Validação aprovada: sinal de apoio preservado, força insuficiente, confiança '    'metodológica criticamente baixa e quatro fontes rastreáveis.'
)

Validação aprovada: sinal de apoio preservado, força insuficiente, confiança metodológica criticamente baixa e quatro fontes rastreáveis.


## Next Steps

A etapa estará validada quando todas as células forem executadas sem erros. O próximo passo será encapsular o fluxo em um serviço de aplicação que receba uma entrada validada e devolva este relatório estruturado.